# Sentiment Analysis Lab Assignment 2

# Name: Bawazir Ahmed Ali Ahmed

# Student ID: SW01084040 

---

In [12]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [13]:
df = pd.read_csv("Reviews.csv")

df = df[['Text', 'Score']]
df = df.dropna()

df = df.sample(10000, random_state=42)

df.head()

,Text,Score
165256,Having tried a couple of other brands of glute...,5
231465,My cat loves these treats. If ever I can't fin...,5
427827,A little less than I expected. It tends to ha...,3
433954,"First there was Frosted Mini-Wheats, in origin...",2
70260,and I want to congratulate the graphic artist ...,5


In [14]:
def get_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Sentiment'] = df['Score'].apply(get_sentiment)

df.head()

,Text,Score,Sentiment
165256,Having tried a couple of other brands of glute...,5,positive
231465,My cat loves these treats. If ever I can't fin...,5,positive
427827,A little less than I expected. It tends to ha...,3,neutral
433954,"First there was Frosted Mini-Wheats, in origin...",2,negative
70260,and I want to congratulate the graphic artist ...,5,positive


In [15]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['Clean_Text'] = df['Text'].apply(clean_text)

df.head()

,Text,Score,Sentiment,Clean_Text
165256,Having tried a couple of other brands of glute...,5,positive,tried couple brands glutenfree sandwich cookie...
231465,My cat loves these treats. If ever I can't fin...,5,positive,cat loves treats ever cant find house pop top ...
427827,A little less than I expected. It tends to ha...,3,neutral,little less expected tends muddy taste expecte...
433954,"First there was Frosted Mini-Wheats, in origin...",2,negative,first frosted miniwheats original size frosted...
70260,and I want to congratulate the graphic artist ...,5,positive,want congratulate graphic artist putting entir...


In [16]:
df.to_csv("cleaned_reviews.csv", index=False)

In [17]:
X = df['Clean_Text']
y = df['Sentiment']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [18]:
bow = CountVectorizer(max_features=3000)
X_train_bow = bow.fit_transform(X_train_text)
X_test_bow = bow.transform(X_test_text)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

y_pred_nb = nb_model.predict(X_test_bow)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8075
              precision    recall  f1-score   support

    negative       0.58      0.61      0.59       280
     neutral       0.23      0.19      0.21       150
    positive       0.89      0.90      0.90      1570

    accuracy                           0.81      2000
   macro avg       0.57      0.57      0.57      2000
weighted avg       0.80      0.81      0.80      2000



In [19]:
tfidf = TfidfVectorizer(max_features=3000)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8355
              precision    recall  f1-score   support

    negative       0.76      0.41      0.54       280
     neutral       0.40      0.03      0.05       150
    positive       0.84      0.99      0.91      1570

    accuracy                           0.84      2000
   macro avg       0.67      0.48      0.50      2000
weighted avg       0.80      0.84      0.79      2000



In [20]:
sia = SentimentIntensityAnalyzer()

def vader_label(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

y_pred_vader = X_test_text.apply(vader_label)

print("VADER Accuracy:", accuracy_score(y_test, y_pred_vader))
print(classification_report(y_test, y_pred_vader))

results = pd.DataFrame({
    'Model': ['Naive Bayes (BoW)', 'Logistic Regression (TF-IDF)', 'VADER'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_vader)
    ]
})

results

VADER Accuracy: 0.7985
              precision    recall  f1-score   support

    negative       0.59      0.34      0.43       280
     neutral       0.08      0.02      0.03       150
    positive       0.83      0.95      0.89      1570

    accuracy                           0.80      2000
   macro avg       0.50      0.44      0.45      2000
weighted avg       0.74      0.80      0.76      2000



,Model,Accuracy
0,Naive Bayes (BoW),0.8075
1,Logistic Regression (TF-IDF),0.8355
2,VADER,0.7985


## Discussion

The three approaches have different strengths and weaknesses for sentiment classification. Naive Bayes with Bag-of-Words is simple, fast, and works well as a baseline, but it treats words independently and ignores context, so it can struggle with subtle meaning. Logistic Regression with TF-IDF usually performs better because TF-IDF gives more informative weights to important words, but it still cannot fully understand sarcasm, word order, or context. VADER is useful because it is a lexicon-based method that does not require model training, making it easy and fast to apply, but it is less adapted to this specific Amazon review dataset and may miss domain-specific language. Overall, Logistic Regression with TF-IDF is the strongest choice here for balanced performance, while Naive Bayes is a good baseline and VADER is a convenient rule-based comparison model.